# Generacion de Texto con LSTM

Modelo de lenguaje: modelo que puede predecir la probabilidad del proximo token dado el anterior.

Flujo:
1. Preparar los **datos de acondicionamiento** (cadena inicial de texto).
2. Generar siguiente token.
3. Añadir token generado a los datos de entrada.
4. Repetir desde el paso 2.

![Diagrama de generacion de tokens a nivel de caracteres de forma iterativa. Encadenando la salida de cada iteracion con la entrada para generar la entrada de la siguiente iteración.](./images/8.1.1.lenguage-model-character-level.png)


Sobre la distribución que genera el modelo de lenguaje hay que elegir el siguiente caracter. A esto se llama *Sampling strategy* (estrategia de muestreo). 

Una estrategia ingenua es siempre elegir el caracter mas probable (*Greedy Sampling*). Pero termina generando cadenas repetitivas y predecibles que no necesariamente se corresponden a un lenguaje coherente.

Una aproximacion mas interesante, es introducir de aleatoriedad proporcional a la distribucion de probabilidad. lo que se llama  *Stochastic Sampling*. Por ejemplo si *e* tiene probabilidad $0.3$ entonces sera escogido el $30\%$ de las veces.

Hay un problema con esa estrategia. No ofrece una forma de controlar la cantidad de aleatoriedad Se quiere un poco para generar secuencias creativas y sorpresivas. Pero no demasiada ya que las secuencias se vuelven incoherentes.

Bajo el orden de **regular la cantidad de entropia** se introduce el parametro ***softmax temperature***. El mismo regula la aleatoridad las distribuciones de probabilidad que luego se usan en la eleccion del proximo token.

Dada un valor de *Temperatura*, se computa una nueva distribución de probabilidad desde la original. A continuacion se muestra un ejemplo de como se realizar esta reponderación.

In [1]:
import numpy as np

def reweight_distribution(original_distribution, temperature=0.5):
    """
    La distribucion original es un arreglo 1D de probabilidades que suman 1. 
    La termperatura cuantifica la entropia de la distribucion.

    Retorna la version reponderada. La suma de la distribucion reponderada es 1.
    """
    distribution = np.log(original_distribution) / temperature
    print(f"Distribution log/temperature: {distribution}")
    distribution = np.exp(distribution)
    print(f"Distribution exp: {distribution}")

    return distribution / np.sum(distribution)

# Ejemplo de uso
distribution = np.array([0.1, 0.2, 0.3, 0.4])
print(f"Original distribution: {distribution}")
rewighted_distribution = reweight_distribution(distribution, temperature=0.5)
print(f"Reweighted distribution: {rewighted_distribution}")

Original distribution: [0.1 0.2 0.3 0.4]
Distribution log/temperature: [-4.60517019 -3.21887582 -2.40794561 -1.83258146]
Distribution exp: [0.01 0.04 0.09 0.16]
Reweighted distribution: [0.03333333 0.13333333 0.3        0.53333333]


A continuacion se muestra un grafico ilustrativo de como las temperaturas bajas convergen la probabilidades en el caracter mas probable. Temperaturas mas altas equiparan todas las probabilidades en un mismo nivel. Mantener la temperatura en 1 hace que la distribucion se mantenga igual.

![Grafico ilustrativo de como las temperaturas bajas convergen la probabilidades en el caracter mas probable. Llevar la temperatura en 1 hace que la distribucion se mantenga igual.](./images/8.1.2.distribution-graph-with-many-temperatures.png)

# Implementacion de LSTM de generación a nivel de caracter

En este ejemplo se usan algunos de los escritos de Nietzsche (filoso de finales del siglo 19). Los textos estan ingles por lo que el modelo entrenado respondera a ese idioma.

El modelo respetara el estilo de escritura de Nietzche y los topicos que manejaba.

## Corpus

A continuacion se muestra como descarcargar los datos.

In [2]:
import keras
import numpy as np

path = keras.utils.get_file(
    'nietzsche.txt',
    origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt',
    cache_dir=".",
    cache_subdir="datasets")
text = open(path, encoding='utf-8').read().lower()
print(f'Corpus length: {len(text)}')

2026-07-31 00:12:00.746081: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-31 00:12:00.746132: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-31 00:12:00.747556: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-31 00:12:00.754902: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Corpus length: 600893


Hay que estandarizar los datos de entrenamiento antes de entrenar el modelo. 

Para ello se van a extraer secuencias parcialmente superpuestas de un largo acotado. Este sera el arreglo `x`.

Tambien se prepara una entrada en un arreglo `y` con los objetivos a predecir por cada `x`.

Posteriormente se aplicara una codificacion *one-hot* a nivel de caracter.

In [3]:
maxlen = 60
step = 3        # Nueva secuencia cada 3 caracteres
sentences = []  # Secuencias extraidas
next_chars = []  # Caracteres siguientes a las secuencias extraidas

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i+maxlen])
    # print(f"> {text[i: i + maxlen]} -> {text[i+maxlen]}")  # Comprobar pares de entrenamiento generados
print(f'Number of sequences: {len(sentences)}')

chars = sorted(list(set(text)))                                     # Lista de caracteres unicos
print(f'Unique characters: {len(chars)}')
char_indices = dict((char, chars.index(char)) for char in chars)    # Diccionario caracter -> indice

print('Vectorización...')
x = np.zeros(
    (len(sentences), maxlen, len(chars)), 
    dtype=bool)
y = np.zeros(
    (len(sentences), len(chars)),
    dtype=bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1
#print(f'x: {x}')
#print(f'y: {y}')

Number of sequences: 200278
Unique characters: 57
Vectorización...


Al tomar secuencias cada 3 caracteres se generaron aproximadamente 1/3 de entradas en relacion al largo total del texto. La cantidad de muestras respeta la siguiente formula:
$$
\text{len(secuencias)} = \left\lfloor \frac{\text{len(text)} - \text{maxlen}}{\text{step}} \right\rfloor
$$

## Construccion

La red constara de una unica capa *LSTM* seguida de un clasificador denso sobre todos los posibles caracteres. 

Como los objetivos a predecir estan en codificacion *one-hot* se utiliza *categorical_crossentropy* como funcion de error.

*Alternativa*: Las redes convolucionales 1D han mostrado un buen rendimiento para este tipo de problemas.

In [4]:
from keras import layers
from keras import optimizers
from keras import models

model = models.Sequential()
model.add(layers.LSTM(128, input_shape=(maxlen, len(chars))))
model.add(layers.Dense(len(chars), activation='softmax'))

optimizer = optimizers.RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

model.summary()

2026-07-31 00:12:06.103013: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-31 00:12:06.109833: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-31 00:12:06.110040: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 128)               95232     
                                                                 
 dense (Dense)               (None, 57)                7353      
                                                                 
Total params: 102585 (400.72 KB)
Trainable params: 102585 (400.72 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Entrenamiento y muestreo

Dado un modelo entrenado y una semilla de texto, se puede generar texto nuevo repitiendo lo siguiente:
1. Generar la distribución de probabilidad del siguiente caracter.
2. Rebalancear la distribucion (con la temperatura especificada).
3. Elegir el proximo caracter acorde a la distribución revalanceada.
4. Añadir el nuevo caracter al final del texto disponible.

A continuacion se muestra el codigo usado para rebalancear la distribucion y elegir el indice de caracter de la misma

In [5]:
import numpy as np

def sample(preds, temperature=1.0):
    """
    La distribucion original es un arreglo 1D de probabilidades que suman 1. 
    La termperatura cuantifica la entropia de la distribucion.

    Retorna un indice de la distribucion reponderada. La suma de la distribucion reponderada es 1.
    """
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

El siguiente bucle entrena y genera texto repetidamente.

Luego de cada epoca se muestra un mismo ejemplo de generacion con distintos rangos de temperatura.

Permite apreciar como evoluciona el texto generado a medida que el modelo converge. 

Ademas del impacto de la temperatura en la estrategia de mustreo.

In [6]:
import random
import sys


for epoch in range(1, 60):
    print(f"epoch {epoch}")
    model.fit(x, y, batch_size=128, epochs=1)                    # Entrena el modelo por 1 iteracion de los datos
    start_index = random.randint(0, len(text) - maxlen - 1)
    generated_text = text[start_index: start_index + maxlen]    # Elige una semilla de texto random
    print(f'--- Generating with seed: "{generated_text}"')

    for temperature in [0.2, 0.5, 1.0, 1.2]:
        print(f'------ temperature: {temperature}')
        sys.stdout.write(generated_text)
        
        for i in range(400):                                    # Genera 400 caracteres desde la semilla de texto
            sampled = np.zeros((1, maxlen, len(chars)))
            for t, char in enumerate(generated_text):           # Codificacion one-hot de la semilla de texto
                sampled[0, t, char_indices[char]] = 1

            preds = model.predict(sampled, verbose=0)[0]
            next_index = sample(preds, temperature)             # Elige el proximo caracter
            next_char = chars[next_index]

            generated_text += next_char
            generated_text = generated_text[1:]                 # Conserva la nueva muestra de longitud maxlen

            sys.stdout.write(next_char)

epoch 1


2026-07-31 00:12:07.070885: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 684950760 exceeds 10% of free system memory.
2026-07-31 00:12:07.632565: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 684950760 exceeds 10% of free system memory.


   1/1565 [..............................] - ETA: 36:52 - loss: 4.0412

2026-07-31 00:12:09.368736: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
2026-07-31 00:12:09.409309: I external/local_xla/xla/service/service.cc:168] XLA service 0x74f9d0edacf0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-31 00:12:09.409349: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA TITAN X (Pascal), Compute Capability 6.1
2026-07-31 00:12:09.415485: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1785456729.461756  683008 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1565/1565 [==============================] - 9s 5ms/step - loss: 2.0236
--- Generating with seed: "greater than suffering.=--there are circumstances in which
s"
------ temperature: 0.2
greater than suffering.=--there are circumstances in which
such a desting the contermand to the conterminess of the precisely the conterning the moral the contermand and still the great the self the contend and still the precisely the contend the contermand the man and still the contermand the great the contention of the contermand the great who desten of the contend the contreater as the conterning the contend of the superity of the contend to the more of------ temperature: 0.5
ng the contend of the superity of the contend to the more of the self thement in the conternity what the desten the
one and man alonistic stapenty to the det to imporise and to
who doont even as contimes and which who he was is the casting of the were resters of the supered to such and contends and the confect of a the is the sci

2026-07-31 00:13:31.402641: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 684950760 exceeds 10% of free system memory.
2026-07-31 00:13:31.919978: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 684950760 exceeds 10% of free system memory.


1565/1565 [==============================] - 8s 5ms/step - loss: 1.6442
--- Generating with seed: "tion"; a cumbersome drapery, something arbitrarily
barbaric "
------ temperature: 0.2
tion"; a cumbersome drapery, something arbitrarily
barbaric of the strange of the subject the consider the same to the end to be delighting the same to any something the same to the same to the operate the sure is the same to the sore of the same the strengther the strength the same the same to the same to the same to the dearness of the end and in the pression that the very profect of the same to an and and the same the man is the sore and the same for th------ temperature: 0.5
 an and and the same the man is the sore and the same for the encontivism the old, it is the end or as an and who exception and strengt the concerning that the for no the word of it is not one, what is the problement to very make all thered is to has he every of themselves of sensision of the existeness of the considers of the vi

2026-07-31 00:14:55.014048: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 684950760 exceeds 10% of free system memory.


1565/1565 [==============================] - 8s 5ms/step - loss: 1.5378
--- Generating with seed: "fficulty being necessary.

62. to be sure--to make also the "
------ temperature: 0.2
fficulty being necessary.

62. to be sure--to make also the sense and conscience and the same that the such a standarily and the same to the self-destrust in the struggle the struggle of the soul--the still the soul the struggle the strict to the standard the strive the same the same to the moral prople and problem the soul. the concerning the struggle and the strict the struggle to the pression of the conscience and some the sense and and the present the ------ temperature: 0.5
f the conscience and some the sense and and the present the comman strom. the does not any plause of the false to the false it weoling the sense any conception of the world to the philosophy is attain have spirit of life it is at the for interpretation of one morality and religious spirit case of the dangerous man conders with th

<img src="images/8.1.3.lstm-nietzsche-train.png" width="800">

A continuacion se muestra el resultado de usar la semilla de texto *"w self esteem seems to him incredible. he can see in it
only"* para generar texto incrementando la temperatura cada 400 caracteres. Esto se hace con el modelo de la epoca 59.

`temperature=0.2`
***
w self esteem seems to him incredible. he can see in it
only to the present religion of the same thing in the same thing in the problem of the belief in the sense of the sense of the sense of the profound thing in the sense of the present the same thing in the state and the sense of the most profound entire vice and and and in the sense of the present and the sense of the entire ter the sense of the same thing in the most contempt and in the sense of the p
***

`temperature=0.5`
***
w self esteem seems to him incredible. he can see in it
he same thing in the most contempt and in the sense of the problem and because the truth to the sense of the instincts, in the past of a more proves himself and his prof in free and the mastery before the imposeness, in which the sense of fact of the case as the result of the sense of the remark and his entire men and recognized themous
and also perhaps spirit as the powerful conscience of the profound continuation of the feeling in him of the sense of whi
***

`temperature=1.0`
***
found continuation of the feeling in him of the sense of which whoe in unable to rank as even their longs that one must, loke that men as the fselve. to power and personness and conessimdifical by this "human predicated difference in its attaces
are, by nature of europe, an inter mirrate, primisis of life--like, ands and in every relatie, fro
marty and inspire the mastery
within himself him
according to its undorge of
making wish as he can realericatiging
***

`temperature=1.2`
***
ing to its undorge of
making wish as he can realericatiging contempt that sin to lights, has only not pase edceation fores; to feel,
undormite oys amd in frances become sympathif
too hagn
problem for poitiousm
of his "plude noely
spirit in resirns.--or menis people as st-presembarinals fastening abacherarred," and sit upon as we knows tostenaver and "most experience,
contend or noble that have natures where chousbinch for schoeen with prides says aod buint
***

Como se aprecia en el texto, la **semántica general** de lo que se genera es **cuestionable**. Sobre todo al compararla con modelos actuales. Aun así, es interesante la generación y, sobre todo, es interesante cómo la temperatura incide en la misma.

- Temperatura **baja** hace el texto **repetible, pero confiable**. En el sentido de que las palabras generadas son sintácticamente correctas.
- Temperaturas **medias** hacen que el texto se repita menos, ya que la aleatoriedad **impide** la persistencia de los **bucles**. A veces se inventan **palabras** que parecieran estar bien escritas, pero no. Incluso podrían sonar **creativas**, por ejemplo, "imposeness" o "personness".
- Temperaturas **altas** generan caos, al punto que el texto parece series de **cadenas semialeatorias**.

Llamo **equilibrio** al punto donde la **temperatura es suficiente para que no haya bucles** y el texto tenga cierta inspiración. Pero **no tan alta como para que sea incoherente**. 

El valor de **equilibrio depende del modelo**. En este caso está cerca de $0.5$. 

A **mayor temperatura peor sintaxis**.

In [7]:
# Guardar el LSTM entrenado
model.save('./models/8.1.lstm_nietzsche.h5')

/workspace/.venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
